# Rain Tomorrow SVM - Hyperparameter Search with MLflow

This notebook performs hyperparameter optimization for the Rain Prediction SVM model
using GridSearchCV and logs all experiments to MLflow.

**Authors:** Paola Blanco, Agustín Vazquez, Facundo Quiroga, Victor Peralta

## Prerequisites
1. Run the `process_etl_rain_data` DAG in Airflow first
2. All services must be running (`docker compose --profile all up`)
3. Set environment variables for MinIO/S3 access

## Steps
1. Load preprocessed data from S3
2. Define SVM hyperparameter search space
3. Run optimization with MLflow tracking
4. Register the best model in MLflow model registry
5. Set the best model as 'champion'

In [ ]:
import os

# Set environment variables for MinIO access
# Change localhost if running on a remote server
os.environ['AWS_ACCESS_KEY_ID'] = 'minio'
os.environ['AWS_SECRET_ACCESS_KEY'] = 'minio123'
os.environ['AWS_ENDPOINT_URL_S3'] = 'http://localhost:9000'
os.environ['MLFLOW_S3_ENDPOINT_URL'] = 'http://localhost:9000'

In [ ]:
import mlflow
import awswrangler as wr
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV, cross_val_score
from sklearn.metrics import (
    f1_score, accuracy_score, precision_score, recall_score,
    confusion_matrix, roc_curve, auc, RocCurveDisplay
)

# MLflow tracking URI (change localhost if needed)
MLFLOW_TRACKING_URI = 'http://localhost:5001'
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

print(f'MLflow tracking URI: {MLFLOW_TRACKING_URI}')
print(f'MLflow version: {mlflow.__version__}')

## 1. Load Data from S3

Load the preprocessed train and test datasets that were created by the ETL DAG.

In [ ]:
# Load train and test data from S3
X_train = wr.s3.read_csv('s3://data/final/train/weather_X_train.csv')
y_train = wr.s3.read_csv('s3://data/final/train/weather_y_train.csv').values.ravel()
X_test = wr.s3.read_csv('s3://data/final/test/weather_X_test.csv')
y_test = wr.s3.read_csv('s3://data/final/test/weather_y_test.csv').values.ravel()

print(f'Training set: {X_train.shape[0]} samples, {X_train.shape[1]} features')
print(f'Test set: {X_test.shape[0]} samples, {X_test.shape[1]} features')
print(f'Target distribution (train): {np.bincount(y_train.astype(int))}')
print(f'Target distribution (test): {np.bincount(y_test.astype(int))}')

## 2. Hyperparameter Search with MLflow Tracking

Define the SVM hyperparameter grid and run GridSearchCV.
Each combination is logged to MLflow for experiment tracking.

In [ ]:
# Create or get the MLflow experiment
experiment = mlflow.set_experiment('Rain Prediction')

# Define hyperparameter grid for SVM
param_grid = {
    'kernel': ['rbf', 'linear', 'poly'],
    'C': [0.1, 1, 10, 100],
    'gamma': ['scale', 'auto', 0.01, 0.1],
}

print('Hyperparameter grid:')
for key, values in param_grid.items():
    print(f'  {key}: {values}')

In [ ]:
# Run GridSearchCV
svm = SVC(probability=True)

grid_search = GridSearchCV(
    estimator=svm,
    param_grid=param_grid,
    cv=5,
    scoring='f1',
    n_jobs=-1,
    verbose=1,
    return_train_score=True
)

grid_search.fit(X_train, y_train)

print(f'\nBest parameters: {grid_search.best_params_}')
print(f'Best CV F1 score: {grid_search.best_score_:.4f}')

## 3. Evaluate Best Model and Log to MLflow

In [ ]:
# Get the best model
best_model = grid_search.best_estimator_

# Evaluate on test set
y_pred = best_model.predict(X_test)

test_f1 = f1_score(y_test, y_pred)
test_accuracy = accuracy_score(y_test, y_pred)
test_precision = precision_score(y_test, y_pred)
test_recall = recall_score(y_test, y_pred)

print(f'Test F1 Score:  {test_f1:.4f}')
print(f'Test Accuracy:  {test_accuracy:.4f}')
print(f'Test Precision: {test_precision:.4f}')
print(f'Test Recall:    {test_recall:.4f}')

In [ ]:
# Log best model to MLflow
with mlflow.start_run(
    run_name='best_svm_model',
    experiment_id=experiment.experiment_id,
    tags={'experiment': 'hyperparameter_search', 'model': 'SVM', 'dataset': 'Rain in Australia'},
) as run:
    # Log parameters
    mlflow.log_params(grid_search.best_params_)
    mlflow.log_param('cv_folds', 5)
    
    # Log metrics
    mlflow.log_metric('f1_score', test_f1)
    mlflow.log_metric('accuracy', test_accuracy)
    mlflow.log_metric('precision', test_precision)
    mlflow.log_metric('recall', test_recall)
    mlflow.log_metric('cv_best_f1', grid_search.best_score_)
    
    # Plot and log confusion matrix
    cm = confusion_matrix(y_test, y_pred)
    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
    ax.set_title('Confusion Matrix - SVM Rain Prediction')
    fig.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
    mlflow.log_artifact('confusion_matrix.png')
    plt.show()
    
    # Plot and log ROC curve
    y_prob = best_model.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    roc_auc = auc(fpr, tpr)
    
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.plot(fpr, tpr, label=f'SVM (AUC = {roc_auc:.4f})')
    ax.plot([0, 1], [0, 1], 'k--')
    ax.set_xlabel('False Positive Rate')
    ax.set_ylabel('True Positive Rate')
    ax.set_title('ROC Curve - SVM Rain Prediction')
    ax.legend()
    fig.savefig('roc_curve.png', dpi=150, bbox_inches='tight')
    mlflow.log_artifact('roc_curve.png')
    mlflow.log_metric('roc_auc', roc_auc)
    plt.show()
    
    # Log the model
    mlflow.sklearn.log_model(best_model, 'svm_rain_prediction')
    
    print(f'\nRun ID: {run.info.run_id}')
    print(f'Experiment ID: {experiment.experiment_id}')

## 4. Register Best Model as Champion

Register the best model in the MLflow model registry and set it as the 'champion'.

In [ ]:
# Register the model
model_name = 'rain_prediction_model_prod'
model_uri = f'runs:/{run.info.run_id}/svm_rain_prediction'

mv = mlflow.register_model(model_uri, model_name)
print(f'Model registered: {model_name}, version: {mv.version}')

# Set the model alias as champion
client = mlflow.MlflowClient()
client.set_registered_model_alias(model_name, 'champion', mv.version)
print(f'Model version {mv.version} set as champion!')

## 5. Verify: Load and Test the Champion Model

In [ ]:
# Verify by loading the champion model
champion_data = client.get_model_version_by_alias(model_name, 'champion')
champion_model = mlflow.sklearn.load_model(champion_data.source)

# Quick test prediction
test_pred = champion_model.predict(X_test[:5])
print(f'Champion model version: {champion_data.version}')
print(f'Test predictions (first 5): {test_pred}')
print(f'Actual values (first 5):    {y_test[:5]}')
print('\nModel is ready to serve via the FastAPI endpoint!')